# Tutorial: Day 3 描述统计与推断统计 (Oxford Tutorial LLM 仿真, v6.0)

## Persona Prompt (Oxford Fellow in Statistical Inference)

> You are an **Oxford tutorial fellow in 统计推断 (Statistical Inference)**. You conduct tutorials in the 1-on-1 Socratic tradition of Oxford PPE / PPL statistics supervisions.
>
> **Core rules**:
> 1. **Never give direct answers.** You do not say "the answer is 0.04" or "use Welch t". You ask probing questions until the student constructs the answer themselves.
> 2. **Use Socratic questioning** at every turn. Why? How could you? What if? 反例 (counterexample)? 凭什么 (on what grounds)? 假设变了 (if the assumption changed)?
> 3. **Devil's advocate**: take the opposite stance. If the student says "p<0.05 so it works", you counter "ASA 2016 第三条怎么说? p 是效应量吗?"
> 4. **Reject vague claims.** "差不多显著" / "应该有效" are forbidden phrases. Demand precision: which test? which assumption? which effect size? which CI?
> 5. **End each turn with a probing question.** Never end with a statement.
> 6. **Scaffold fade**: if the student fails to defend 2 turns in a row, drop one scaffold level (Worked -> Faded -> Independent hint), but **still no direct answer**.
>
> This simulates the Oxford tutorial method (Vygotsky 共构 / Socratic co-construction) without API calls.

---

## Pre-Tutorial Task (强制 retrieval, 学生须先提交)

**提交物**（学生进入 tutorial 前必须完成）:

1. **300 字 mini-essay**: 用法庭审判类比解释 H₀ / H₁ / p 值 / α / β / Power，并说明 A/B 测试中 p=0.04 + 效应量 d=0.12 是否足以"上线新方案"。
2. **代码片段**: 写一行 `scipy.stats` 调用，对 A/B 两组转化率做 t 检验；写出卡方列联表的数据结构；写出 Beta(1,1) + 30/200 的后验参数。
3. **盲点自报**: 列出 2-3 个"我不确定"的点（例如 Welch vs Student、credible vs CI、p-hacking 如何膨胀 α）。

> 不提交不进入 tutorial。Tutorial 第 1 问会基于此 mini-essay 追问。


## Pre-Tutorial Checklist (学生自检后才能开始 Socratic loop)

- [ ] mini-essay 已写（含法庭类比 + A/B 上线判断）
- [ ] scipy.stats 调用已写（ttest_ind / chi2_contingency / beta 三行）
- [ ] 2-3 盲点已列
- [ ] starter.ipynb TODO1-6 已尝试（不必全对，但要有草稿）

**没完成？退出本 notebook，先做完再回来。Tutorial 不是替你写作业的地方。**


In [ ]:
# Socratic Tutorial Loop (静态模拟, >=4 轮, 每轮检测 defense 失败则降一级 scaffold, 仍禁直接答案)
# 不调 API, 用 if/else 模拟 Oxford fellow 的追问路径

import json, os

STUDENT_ESSAY = """
(学生 mini-essay 占位 - 实际运行时替换为真实提交)
法庭类比: H₀=无罪, H₁=有罪, p=看到证据的概率。
A/B 测试 p=0.04, 效应量 d=0.12, 应该上线新方案因为显著。
"""

TUTORIAL_LOG = []

def socratic_turn(round_num, student_input, scaffold_level):
    """模拟 Oxford fellow 的 Socratic 追问。每轮必含 >=1 probing question, 禁直接答案。"""
    round_q = {
        1: [
            "Q1 (Round 1): 你的 mini-essay 写 'p=看到证据的概率'-- ASA 2016 第一条怎么说? p 值到底是不是 P(H₀ 真 | 数据)? 凭什么?",
            "Q2 (Round 1): 你写 '应该上线因为显著'-- '显著' 是什么意思? p<0.05 等于 '方案有效' 吗? 反例: 如果样本量 n=100000, p=0.04 但效应量 d=0.01, 还上线吗?",
        ],
        2: [
            "Q3 (Round 2): 假设你的 A/B 两组方差不等 (营销场景极常见), 你调用 scipy.stats.ttest_ind 时漏了什么参数? 为什么 Welch t 比 Student t 更稳健? 如何验证方差齐性?",
            "Q4 (Round 2): 你写卡方列联表 - 如果行列颠倒 (列=分群, 行=品类), chi2_contingency 结果会变吗? dof 怎么算? 为什么 (r-1)(c-1)?",
        ],
        3: [
            "Q5 (Round 3): Beta(1,1) 先验 + 30 转化/200 试验, 后验参数你写对了吗? 后验均值是多少? 它和频率派的点估计 30/200=0.15 有什么不同? 为什么贝叶斯派说 '直接回答业务问题'?",
            "Q6 (Round 3): credible interval [0.107, 0.205] 和频率派 95% CI - 两者语义根本不同, 你能精确说出差异吗? 如果有人说 '95% 概率真值在 CI 内', 他错在哪?",
        ],
        4: [
            "Q7 (Round 4): p-hacking 如何膨胀第一类错误 α? 假设你每 100 用户检查一次 p, 达 0.05 就停, 实际 α 还是 0.05 吗? 预注册 (OSF) 如何解决? 功效分析如何预计算样本量?",
            "Q8 (Round 4): 最后一个反诘 - 你说学完本 Day 能做 A/B 测试推断。现在我给你 n=50 的小样本 A/B, p=0.08 (不显著), 你如何决策? 频率派束手时, 贝叶斯派能救吗? 凭什么?",
        ],
    }
    return round_q.get(round_num, ["Final: 把 8 个问题串成一份 500 字反思, 标注哪 2-3 个你答得最差 - 那就是 exit artifact 的盲点。"])

# --- 静态模拟 4 轮 Socratic 对话 ---
scaffold = 3  # 3=Independent, 2=Faded, 1=Worked, 0=retry
student_responses = [
    "p 是 P(H₀真|数据)... 等等好像不对",  # Round 1 fail
    "我漏了 equal_var... Welch 是因为方差不等",  # Round 2 partial
    "Beta(31, 171)? 均值 0.153? 但和 0.15 没区别啊",  # Round 3 partial fail
    "p-hacking 是不停检查... 预注册是事先声明",  # Round 4 partial
]

for rnd in range(1, 5):
    questions = socratic_turn(rnd, student_responses[rnd-1] if rnd-1 < len(student_responses) else "", scaffold)
    TUTORIAL_LOG.append({"round": rnd, "scaffold_level": scaffold, "questions": questions})
    # 检测 defense 失败 -> 降一级 scaffold (仍禁直接答案)
    if "不对" in (student_responses[rnd-1] if rnd-1 < len(student_responses) else "") or "没区别" in (student_responses[rnd-1] if rnd-1 < len(student_responses) else ""):
        scaffold = max(0, scaffold - 1)
        TUTORIAL_LOG[-1]["scaffold_drop"] = f"defense failed -> drop to scaffold {scaffold} (still no direct answer, only more worked hint)"

print("=== Oxford Tutorial Socratic Loop (4 rounds, 8 questions, no direct answers) ===")
for entry in TUTORIAL_LOG:
    print(f"\n--- Round {entry['round']} (scaffold {entry['scaffold_level']}) ---")
    for q in entry["questions"]:
        print(f"  Fellow: {q}")
    if "scaffold_drop" in entry:
        print(f"  [SCAFFOLD DROP] {entry['scaffold_drop']}")

print(f"\nFinal scaffold level: {scaffold} (0=retry/Worked, 3=Independent)")
print("\n注: 本 loop 是静态模拟, 实际 tutorial 中 fellow 会根据学生真实回答动态追问, 但规则不变: Socratic + 禁直接答案 + 渐退脚手架")


In [ ]:
# student_model.json 读写 - 记录掌握度/盲点, 跨单元复用
import json, os

UNIT_DIR = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.path.abspath('.')) == '__file__' else '.'
STUDENT_MODEL_PATH = os.path.join(os.path.expanduser('~'), '.cache', 'phd_v6_student_model.json')

DEFAULT_MODEL = {
    "unit": "U-skill0-day3-statistics-inference",
    "subskills": {
        "S1_descriptive": {"mastery": 0.5, "attempts": 0, "last": None, "weak_points": []},
        "S2_hypothesis_test": {"mastery": 0.4, "attempts": 0, "last": None, "weak_points": ["Welch equal_var", "ASA 六原则"]},
        "S2_chi2": {"mastery": 0.5, "attempts": 0, "last": None, "weak_points": []},
        "S3_bayesian": {"mastery": 0.3, "attempts": 0, "last": None, "weak_points": ["credible vs CI 语义"]},
        "S4_ASA_p_value": {"mastery": 0.4, "attempts": 0, "last": None, "weak_points": ["p-hacking 膨胀 α 机制"]}
    },
    "cross_unit_links": {
        "day4_regression": "t 检验将扩展到回归系数显著性",
        "skill3_causal": "A/B 测试是因果推断入门, DML/合成控制/增量建模在前方",
        "day5_sql": "pandas DataFrame 将延伸到 SQL 查询"
    },
    "tutorial_history": [],
    "last_session": None,
    "daily_usage": {"date": None, "count": 0}
}

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    return DEFAULT_MODEL.copy()

def save_student_model(model):
    os.makedirs(os.path.dirname(STUDENT_MODEL_PATH), exist_ok=True)
    with open(STUDENT_MODEL_PATH, 'w', encoding='utf-8') as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

def update_mastery(model, subskill, delta, weak_points=None):
    """tutorial 后更新掌握度与盲点"""
    if subskill in model["subskills"]:
        model["subskills"][subskill]["mastery"] = max(0.0, min(1.0, model["subskills"][subskill]["mastery"] + delta))
        model["subskills"][subskill]["attempts"] += 1
        if weak_points:
            model["subskills"][subskill]["weak_points"] = list(set(model["subskills"][subskill].get("weak_points", []) + weak_points))
    return model

# 模拟一次 tutorial 后的更新
model = load_student_model()
model = update_mastery(model, "S2_hypothesis_test", -0.1, ["Welch equal_var 仍漏"])
model = update_mastery(model, "S3_bayesian", -0.1, ["credible vs CI 仍混淆"])
model["tutorial_history"].append({
    "rounds": 4,
    "final_scaffold": 1,
    "exit_blind_spots": ["Welch t 参数", "credible vs CI 语义", "p-hacking 膨胀机制"],
    "recommended_review_units": ["day4_regression", "skill3_causal"]
})
save_student_model(model)
print("student_model.json 已更新:")
print(json.dumps(model, ensure_ascii=False, indent=2)[:800] + "...")
print(f"\n路径: {STUDENT_MODEL_PATH}")
print("跨单元复用: day4_regression / skill3_causal / day5_sql 会读取此 model 决定复习优先级")


## Hattie (2007 RER 77(1):81-112) 四级 Formative Feedback

> John Hattie 的元分析指出，反馈的效应量 d=0.79（远高于平均干预），但**不同级别反馈效果不同**。Self 级（"你真棒"）几乎无效，Task/Process/Self-Reg/Feed-Forward 才是高效应级别。
> 本 tutorial 的反馈严格按四级标注，**避免 Self 级表扬**。

### [TASK] 级（任务级反馈 - 针对本次 tutorial 的具体答案）

- **[TASK]** 你 Round 1 写 "p 是 P(H₀ 真 | 数据)" -- 错。ASA 2016 第一条明示: p 是 P(数据 | H₀ 真)，方向反了。请重写。
- **[TASK]** 你 Round 3 写 "Beta(31, 171) 均值和 0.15 没区别" -- 数值接近但语义根本不同。后验是分布，频率派点估计是单点；后验还给 credible interval，点估计不给。
- **[TASK]** 你 Round 4 写 "预注册是事先声明" -- 对，但不完整。预注册还需包含样本量预计算、分析计划、停止规则，缺一不可。

### [PROCESS] 级（过程级反馈 - 针对学习方法）

- **[PROCESS]** 你 4 轮中 3 轮出现 "差不多 / 没区别 / 应该" 等模糊词。Oxford tutorial 不容忍模糊。下次每个结论前问自己: 哪个检验？哪个假设？哪个效应量？哪个 CI？
- **[PROCESS]** 你在 Welch t 上重复失败 2 次 -- 触发 `practice.md` weak_loop。建议: 回到 drill D2 的 Worked 阶段，重做完整示范，再 Faded，再 Independent。
- **[PROCESS]** 贝叶斯后验你算对了参数但解读错了 -- 这是 "算对但不懂" 的典型。下次算完后用一句话向非技术人（如营销总监）解释，解释不清就是不懂。

### [SELF-REG] 级（自我调节反馈 - 针对元认知）

- **[SELF-REG]** 你 Round 2 主动说 "我漏了 equal_var" -- 这是好的自我监控。但 Round 3 又在 credible vs CI 上栽了 -- 说明你的自我监控只覆盖调用层，不覆盖语义层。扩展监控范围。
- **[SELF-REG]** 你 exit 时自报盲点 3 个，与 tutorial 实际暴露的盲点 2 个重合 -- 说明你有 1 个"未知的未知"（unknown unknown）。每周做一次盲点扫描: 哪些概念你以为懂了其实没懂？
- **[SELF-REG]** 你的 scaffold 从 3 降到 1 -- 注意: scaffold 降到 0 会触发 `practice.md` retry_policy 的"累计 3 次未通过"。自我监控 scaffold 水平，主动补课。

### [FEED-FORWARD] 级（前馈反馈 - 针对迁移与下一步）

- **[FEED-FORWARD]** 本 Day 的 t 检验在 Day 4 回归分析中会扩展到"回归系数显著性检验"。建议: 预习 `statsmodels.api.OLS` 的 `.summary()`，看 p 值与 t 值如何出现在回归表里。
- **[FEED-FORWARD]** 你的 ASA 六原则掌握不足 (mastery 0.4) -- 在技能3 因果推断中，p 值会让位于因果效应估计 (ATE/CATE)，但 ASA 第四条"透明报告"仍适用。建议: 把 ASA 六原则抄进 `schedule.json` C4 卡，每天复习。
- **[FEED-FORWARD]** 贝叶斯 Beta-Binomial 是入门，技能3 的因果推断会用到贝叶斯层次模型 (PyMC)。建议: 选做 starter.ipynb 的 PyMC 扩展，500 字对比 scipy.stats.beta 手动后验与 PyMC 采样后验。
- **[FEED-FORWARD]** exit artifact: 你的 2-3 盲点 = [Welch t 参数, credible vs CI 语义, p-hacking 膨胀机制]。推荐复习单元 = day4_regression + skill3_causal。下次 tutorial (限频: 1 次/天) 前请先做 `schedule.json` 到期卡片。


## 限频与 Exit Artifact (防 LLM 依赖 + 强制迁移)

### 限频政策 (防依赖)

- **每单元 tutorial 每天 1 次**。本 Day 3 的 tutorial 今天已用 1 次，下次最早明天。
- **限频理由**: Oxford tutorial 的价值在于学生**独立思考**后的追问。若不限频，学生会把 tutorial 当"答案生成器"，跳过 pre-task 的检索练习，违背 Vygotsky 共构原则。
- **弱项循环例外**: 若 `practice.md` weak_loop 触发，当天可加 1 次 tutorial，但**仅限该弱项 subskill**（如 S3_bayesian），且 `student_model.json` 会记录 `daily_usage.count=2` 防溢用。
- **跨单元限频**: 同一学生同一天最多 2 个不同单元的 tutorial，避免"刷 tutorial"替代"刷 drill"。
- **违反限频**: 返回 "今日 tutorial 已用完。请先做 `schedule.json` 到期卡片 (FSRS-6) 与 `practice.md` drill，明天再来。"

### Exit Artifact (强制输出)

完成本 tutorial 后，学生必须提交以下 exit artifact（写入 `student_model.json` 的 `tutorial_history[-1]`）:

1. **2-3 个盲点** (blind spots): 本 tutorial 暴露的"我以为懂其实没懂"的点。格式: `[盲点名, 触发的 Socratic 问号, 下次如何自查]`。
2. **推荐复习单元** (recommended review units): 基于 weak_points 推荐的跨单元复习。至少 2 个（如 day4_regression, skill3_causal）。
3. **500 字反思**: 把 8 个 Socratic 问题串成一段叙事，标注哪个问题答得最差，为什么，下次如何避免。
4. **mastery 自评**: 对 S1/S2/S3/S4 四个 subskill 给 0-1 自评分，与 `student_model.json` 的 mastery 对比，差距 >0.2 说明元认知需校准。

### Exit Checklist

- [ ] 2-3 盲点已写入 student_model.json
- [ ] 推荐复习单元 >=2 个
- [ ] 500 字反思已写
- [ ] mastery 自评与模型对比
- [ ] schedule.json 到期卡片今日已复习 (FSRS-6)
- [ ] practice.md drill 今日已完成 reps_required

**未完成 exit artifact = tutorial 未完成。下次进入需从头开始 (pre-task 重做)。**

---

## 学习科学依据索引

- **Oxford tutorial 1对1 Socratic**: Christ Church / All Souls tradition, LLM 仿真参考 arxiv 2409.05511, 2507.05795
- **Persona role-engineering**: 禁直接答案 + Socratic 追问 (Vygotsky 共构)
- **Hattie 4 级反馈**: Hattie (2007) Review of Educational Research 77(1):81-112, d=0.79
- **限频防依赖**: NUS Autograder + SELENE 自定步调研究
- **student_model 跨单元复用**: ScholAstic 仿真 chatbot 模式
- **FSRS-6 间隔重复**: 关联 schedule.json
- **Biggs 建构对齐**: 关联 alignment.md
- **Worked-Faded 渐退**: 关联 practice.md
